In [1]:
import os
os.environ["PYSPARK_PYTHON"] = "C:/Users/complere/anaconda3/envs/pyspark_env/python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = "C:/Users/complere/anaconda3/envs/pyspark_env/python.exe"

In [2]:
import pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("test").getOrCreate()

In [3]:
df1 = spark.createDataFrame([(1, "Alice"),(2, "Bob"),(3, "Charlie")], 
                            ["id", "name"])

In [4]:
df2 = spark.createDataFrame([(1, "HR"),(2, "Finance"),(4, "Marketing")], ["id", "department"])

In [5]:
df1.show()

+---+-------+
| id|   name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
+---+-------+



In [6]:
df2.show()

+---+----------+
| id|department|
+---+----------+
|  1|        HR|
|  2|   Finance|
|  4| Marketing|
+---+----------+



# Inner Join

In [8]:
df_inner = df1.join(df2, on="id", how="inner")
df_inner.show()

+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1|Alice|        HR|
|  2|  Bob|   Finance|
+---+-----+----------+



In [9]:
#Returns only matching records.

df1.join(df2, df1.id == df2.id, "inner").show()

+---+-----+---+----------+
| id| name| id|department|
+---+-----+---+----------+
|  1|Alice|  1|        HR|
|  2|  Bob|  2|   Finance|
+---+-----+---+----------+



# Left Join / Left Outer Join

In [13]:
df1.join(df2,df1.id == df2.id, "left").show()

+---+-------+----+----------+
| id|   name|  id|department|
+---+-------+----+----------+
|  1|  Alice|   1|        HR|
|  2|    Bob|   2|   Finance|
|  3|Charlie|NULL|      NULL|
+---+-------+----+----------+



In [10]:
#Returns all rows from the left DataFrame and matches from the right.
#If there is a matching row in the right DataFrame (df2), the values from df2 are added.
#If no match exists, Spark fills the right DataFrame’s columns with null.

df1.join(df2, "id", "left").show()

+---+-------+----------+
| id|   name|department|
+---+-------+----------+
|  1|  Alice|        HR|
|  2|    Bob|   Finance|
|  3|Charlie|      NULL|
+---+-------+----------+



In [12]:
df_left = df1.join(df2, on="id", how="left")
df_left.show()


+---+-------+----------+
| id|   name|department|
+---+-------+----------+
|  1|  Alice|        HR|
|  2|    Bob|   Finance|
|  3|Charlie|      NULL|
+---+-------+----------+



# Right Join / Right Outer Join

In [14]:
#Returns all rows from the right DataFrame.
#If a matching row exists in the left DataFrame (df1), values from the left are included.
#If no match exists, the left DataFrame’s columns are filled with null.

df1.join(df2, "id", "right").show()

+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1|Alice|        HR|
|  2|  Bob|   Finance|
|  4| NULL| Marketing|
+---+-----+----------+



# Full Outer Join

In [15]:
#Returns all rows from both DataFrames.
#If a row exists in both DataFrames, values are combined.
#If a row exists in only one DataFrame, the columns from the other DataFrame are filled with null.

df1.join(df2, "id", "outer").show()

+---+-------+----------+
| id|   name|department|
+---+-------+----------+
|  1|  Alice|        HR|
|  2|    Bob|   Finance|
|  3|Charlie|      NULL|
|  4|   NULL| Marketing|
+---+-------+----------+



In [18]:
df_outer = df1.join(df2, on="id", how="outer")
df_outer.show()

+---+-------+----------+
| id|   name|department|
+---+-------+----------+
|  1|  Alice|        HR|
|  2|    Bob|   Finance|
|  3|Charlie|      NULL|
|  4|   NULL| Marketing|
+---+-------+----------+



# Left Semi Join

In [ ]:
##Returns only left rows that have a match in right, but no columns from right.

In [7]:
df1.show()

+---+-------+
| id|   name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
+---+-------+



In [8]:
df2.show()

+---+----------+
| id|department|
+---+----------+
|  1|        HR|
|  2|   Finance|
|  4| Marketing|
+---+----------+



In [9]:
df1.join(df2, "id", "left_semi").show()

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



# Left Anti Join

In [22]:
#Returns left rows that do NOT match in right.
#Essentially, it filters out any rows from the left that exist in the right based on the key.

df1.join(df2, "id", "left_anti").show() # A-B 

+---+-------+
| id|   name|
+---+-------+
|  3|Charlie|
+---+-------+



# Cross Join (Cartesian)

In [23]:
#Creates every combination (⚠️ huge output).

df1.crossJoin(df2).show()

+---+-------+---+----------+
| id|   name| id|department|
+---+-------+---+----------+
|  1|  Alice|  1|        HR|
|  1|  Alice|  2|   Finance|
|  1|  Alice|  4| Marketing|
|  2|    Bob|  1|        HR|
|  2|    Bob|  2|   Finance|
|  2|    Bob|  4| Marketing|
|  3|Charlie|  1|        HR|
|  3|Charlie|  2|   Finance|
|  3|Charlie|  4| Marketing|
+---+-------+---+----------+



# Aggregations in PySpark


In [ ]:
# groupBy, agg, countDistinct
#Common operations: grouping, averaging, counting, finding max/min values.



In [10]:
from pyspark.sql.functions import sum, avg, max, min, count,col

In [11]:
df1 = spark.createDataFrame([
    ("HR", 50000),
    ("HR", 60000),
    ("IT", 70000),
    ("IT", 65000),
    ("Finance", 55000)
], ["department", "salary"])

In [12]:
df1.show()

+----------+------+
|department|salary|
+----------+------+
|        HR| 50000|
|        HR| 60000|
|        IT| 70000|
|        IT| 65000|
|   Finance| 55000|
+----------+------+



In [13]:
df1.printSchema()

root
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)



# Sum

In [15]:
df.groupby("department")["salary"].sum()   # as 

TypeError: 'GroupedData' object is not subscriptable

In [9]:
df2=df1.groupBy("department").agg(sum("salary").alias("total_salary")) # alias to rename column

In [10]:
df2.show()

+----------+------------+
|department|total_salary|
+----------+------------+
|        HR|      110000|
|        IT|      135000|
|   Finance|       55000|
+----------+------------+



In [7]:
df1.groupBy("department").agg(sum("salary")).show() 

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|        HR|     110000|
|        IT|     135000|
|   Finance|      55000|
+----------+-----------+



In [ ]:
# in pandas
df.groupby("department")["salary"].sum().reset_index(name="total_salary")

               OR

result = df.groupby("department")["salary"].sum().reset_index()
result.columns = ["department", "total_salary"]

print(result)


In [34]:
df2 = df.groupBy("department").agg(sum("salary").alias("total_salary"))

In [35]:
df2.show()

+----------+------------+
|department|total_salary|
+----------+------------+
|        HR|      110000|
|        IT|      135000|
|   Finance|       55000|
+----------+------------+



# add a new column 

In [36]:
from pyspark.sql.functions import col
df2 = df2.withColumn("bonus", col("total_salary") * 0.10)

df2.show()

+----------+------------+-------+
|department|total_salary|  bonus|
+----------+------------+-------+
|        HR|      110000|11000.0|
|        IT|      135000|13500.0|
|   Finance|       55000| 5500.0|
+----------+------------+-------+



In [6]:
df.show()

+----------+------+
|department|salary|
+----------+------+
|        HR| 50000|
|        HR| 60000|
|        IT| 70000|
|        IT| 65000|
|   Finance| 55000|
+----------+------+



In [68]:
#Multiple Aggregations Together
df.groupBy("department").agg(
    sum("salary").alias("total"),
    avg("salary").alias("avg"),
    max("salary").alias("max"),
    min("salary").alias("min")
).show()

+----------+------+-------+-----+-----+
|department| total|    avg|  max|  min|
+----------+------+-------+-----+-----+
|        HR|110000|55000.0|60000|50000|
|        IT|135000|67500.0|70000|65000|
|   Finance| 55000|55000.0|55000|55000|
+----------+------+-------+-----+-----+



In [8]:
df.show()

+----------+------+
|department|salary|
+----------+------+
|        HR| 50000|
|        HR| 60000|
|        IT| 70000|
|        IT| 65000|
|   Finance| 55000|
+----------+------+



In [67]:
#Global Aggregations (No GroupBy)
df.agg(
    sum("salary"),
    avg("salary"),
    count("*")
).show()

+-----------+-----------+--------+
|sum(salary)|avg(salary)|count(1)|
+-----------+-----------+--------+
|     300000|    60000.0|       5|
+-----------+-----------+--------+



In [75]:
#Aggregation with Filtering (having clause)
#PySpark doesn’t have HAVING, but we can  filter after aggregation:
# having avg_salary>60000

df.groupBy("department").agg(avg("salary").alias("avg_salary")).filter("avg_salary > 60000").show()



+----------+----------+
|department|avg_salary|
+----------+----------+
|        IT|   67500.0|
+----------+----------+



# drop duplicate

In [60]:
df_distinct = df.dropDuplicates()
df_distinct.show()

+---+----+
| id|name|
+---+----+
|  1|   A|
|  2|   B|
|  3|   C|
+---+----+



In [61]:
#Remove duplicates based on selected columns (example: id only)
df_distinct = df.dropDuplicates(["id"])
df_distinct.show()


+---+----+
| id|name|
+---+----+
|  1|   A|
|  2|   B|
|  3|   C|
+---+----+



In [ ]:
#Keep FIRST or LAST duplicate
#PySpark doesn’t have keep='first' by default, so we use window functions.

# Pivot (Dynamic Aggregation)

In [12]:
data = [
    (1, "Alice", "HR", 5000, 500, 2022),
    (2, "Bob", "IT", 6000, 600, 2021),
    (3, "Charlie", "IT", 7000, 700, 2022),
    (4, "David", "Finance", 5200, 520, 2022),
    (5, "Eva", "HR", 6000, 600, 2021),
]

columns = ["id", "name", "department", "salary", "bonus", "year"]

df = spark.createDataFrame(data, columns)
df.show()

+---+-------+----------+------+-----+----+
| id|   name|department|salary|bonus|year|
+---+-------+----------+------+-----+----+
|  1|  Alice|        HR|  5000|  500|2022|
|  2|    Bob|        IT|  6000|  600|2021|
|  3|Charlie|        IT|  7000|  700|2022|
|  4|  David|   Finance|  5200|  520|2022|
|  5|    Eva|        HR|  6000|  600|2021|
+---+-------+----------+------+-----+----+



In [84]:
#Pivot by year and sum salary (Row --------------> Column)
pivot_df = df.groupBy("department").pivot("year").agg(sum("salary"))
pivot_df.show()

+----------+----+----+
|department|2021|2022|
+----------+----+----+
|        HR|6000|5000|
|   Finance|NULL|5200|
|        IT|6000|7000|
+----------+----+----+



#  Count Distinct

In [ ]:
The countDistinct() function (from pyspark.sql.functions) is an aggregation function 
used to count the number of unique, non-null values in a specified column used  with group by.


In [13]:
data = [
    (1, "Alice", "HR"),
    (2, "Bob", "IT"),
    (3, "Charlie", "IT"),
    (1, "Alice",None),     # duplicate employee in HR
    (4, "David", "Finance"),
    (5, "Eva", "HR"),
    (2, "Bob", "IT"),       # duplicate employee in IT
]

columns = ["employee_id", "name", "department"]

df = spark.createDataFrame(data, columns)
df.show()

+-----------+-------+----------+
|employee_id|   name|department|
+-----------+-------+----------+
|          1|  Alice|        HR|
|          2|    Bob|        IT|
|          3|Charlie|        IT|
|          1|  Alice|      NULL|
|          4|  David|   Finance|
|          5|    Eva|        HR|
|          2|    Bob|        IT|
+-----------+-------+----------+



In [14]:
df.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    2|
|        IT|    3|
|      NULL|    1|
|   Finance|    1|
+----------+-----+



In [ ]:
#Count distinct employees per department

In [15]:
from pyspark.sql.functions import countDistinct
df.groupBy("department").agg(countDistinct("employee_id").alias("num_employees")).show()

+----------+-------------+
|department|num_employees|
+----------+-------------+
|        HR|            2|
|      NULL|            1|
|   Finance|            1|
|        IT|            2|
+----------+-------------+



In [ ]:
# Explanation
HR has employees: 1 (Alice), 5 (Eva) → 2 unique

IT has employees: 2 (Bob), 3 (Charlie) → 2 unique

Finance has employee: 4 (David) → 1 unique

In [16]:
df.groupBy("department").agg(countDistinct("name").alias("unique_names")).show()

+----------+------------+
|department|unique_names|
+----------+------------+
|        HR|           2|
|      NULL|           1|
|   Finance|           1|
|        IT|           2|
+----------+------------+



In [ ]:
#isNull()
df[df['salary'].isnull()]
df.salary.isnull()
#Filter rows where salary is NULL:

In [18]:
df.show()

+-----------+-------+----------+
|employee_id|   name|department|
+-----------+-------+----------+
|          1|  Alice|        HR|
|          2|    Bob|        IT|
|          3|Charlie|        IT|
|          1|  Alice|      NULL|
|          4|  David|   Finance|
|          5|    Eva|        HR|
|          2|    Bob|        IT|
+-----------+-------+----------+



In [19]:
df.filter(col("department").isNull()).show()

+-----------+-----+----------+
|employee_id| name|department|
+-----------+-----+----------+
|          1|Alice|      NULL|
+-----------+-----+----------+



In [20]:
df.filter(col("department").isNotNull()).show()

+-----------+-------+----------+
|employee_id|   name|department|
+-----------+-------+----------+
|          1|  Alice|        HR|
|          2|    Bob|        IT|
|          3|Charlie|        IT|
|          4|  David|   Finance|
|          5|    Eva|        HR|
|          2|    Bob|        IT|
+-----------+-------+----------+



In [52]:
df.filter(col("name").isin("Alice", "Bob")).show()

+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|Alice|  5000|
|  2|  Bob|  NULL|
+---+-----+------+



In [53]:
df.filter(col("salary").between(5000, 8000)).show()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  1|  Alice|  5000|
|  3|Charlie|  7000|
+---+-------+------+



In [54]:
df.filter((col("salary") > 5000) & (col("name").startswith("A"))).show()

+---+----+------+
| id|name|salary|
+---+----+------+
|  5|Alan|  9000|
+---+----+------+



In [55]:
df.filter((col("salary") > 5000) | (col("name") == "Bob")).show()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  2|    Bob|  NULL|
|  3|Charlie|  7000|
|  5|   Alan|  9000|
+---+-------+------+



In [56]:
df.filter("name LIKE 'A%' AND salary IS NOT NULL").show()

+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|Alice|  5000|
|  5| Alan|  9000|
+---+-----+------+



# 2ND PART

In [ ]:
#Key Components of a Window
Partitioning → defines groups of data (like SQL PARTITION BY).
Ordering → defines row order inside each partition.
Frame → defines the range of rows considered for each calculation.


In [1]:
import os
os.environ["PYSPARK_PYTHON"] = "C:/Users/complere/anaconda3/envs/pyspark_env/python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = "C:/Users/complere/anaconda3/envs/pyspark_env/python.exe"

In [2]:
import pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("test").getOrCreate()

In [20]:
data = [
    (1, "Alice", "HR", 5000),
    (2, "Bob", "IT", 6000),
    (3, "Charlie", "IT", 7000),
    (4, "David", "Finance", 5200),
    (5, "Eva", "HR", 6000),
    (6, "Frank", "Finance", 4800),
    (7, "renu", "Finance", 4900),
    (8, "renu", "Finance", 4700)
]

columns = ["id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.show()

+---+-------+----------+------+
| id|   name|department|salary|
+---+-------+----------+------+
|  1|  Alice|        HR|  5000|
|  2|    Bob|        IT|  6000|
|  3|Charlie|        IT|  7000|
|  4|  David|   Finance|  5200|
|  5|    Eva|        HR|  6000|
|  6|  Frank|   Finance|  4800|
|  7|   renu|   Finance|  4900|
|  8|   renu|   Finance|  4700|
+---+-------+----------+------+



In [21]:
#Import Window Functions
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, row_number, dense_rank, sum, avg, lag, lead,col,min,max,count

In [ ]:
# in sql
select *,sum(salary) over (partition by dept )  as total_sal_per_dept from table;
select *,count(id) over (partition by dept )  as total_count_per_dept from table;


In [22]:
df.withColumn("count_dept", count("id").over(Window.partitionBy("department"))).show()

+---+-------+----------+------+----------+
| id|   name|department|salary|count_dept|
+---+-------+----------+------+----------+
|  4|  David|   Finance|  5200|         4|
|  6|  Frank|   Finance|  4800|         4|
|  7|   renu|   Finance|  4900|         4|
|  8|   renu|   Finance|  4700|         4|
|  1|  Alice|        HR|  5000|         2|
|  5|    Eva|        HR|  6000|         2|
|  2|    Bob|        IT|  6000|         2|
|  3|Charlie|        IT|  7000|         2|
+---+-------+----------+------+----------+



In [ ]:
windowSpec=Window.partitionBy("department"))

In [9]:
df.withColumn("count_dept", count("id").over(windowSpec)) \
  .withColumn("min_salary", min("salary").over(windowSpec)) \
  .withColumn("max_salary", max("salary").over(windowSpec)) \
  .withColumn("avg_salary", avg("salary").over(windowSpec)) \
  .show()

+---+-------+----------+------+----------+----------+----------+----------+
| id|   name|department|salary|count_dept|min_salary|max_salary|avg_salary|
+---+-------+----------+------+----------+----------+----------+----------+
|  4|  David|   Finance|  5200|         4|      4700|      5200|    4900.0|
|  6|  Frank|   Finance|  4800|         4|      4700|      5200|    4900.0|
|  7|   renu|   Finance|  4900|         4|      4700|      5200|    4900.0|
|  8|   renu|   Finance|  4700|         4|      4700|      5200|    4900.0|
|  1|  Alice|        HR|  5000|         2|      5000|      6000|    5500.0|
|  5|    Eva|        HR|  6000|         2|      5000|      6000|    5500.0|
|  2|    Bob|        IT|  6000|         2|      6000|      7000|    6500.0|
|  3|Charlie|        IT|  7000|         2|      6000|      7000|    6500.0|
+---+-------+----------+------+----------+----------+----------+----------+



# RANK 

In [ ]:
select *,rank() over (partition by dept order by salary desc )  as rank_per_dept_salary from table;

select *,dense_rank() over (partition by dept order by salary desc )  as rank_per_dept_salary from table;
select *,row_number() over (partition by dept order by salary desc )  as rank_per_dept_salary from table;


In [26]:

# Partition by department and order by salary descending
windowSpec = Window.partitionBy("department").orderBy(col("salary"))


In [ ]:
# explanation
partitionBy("department") → calculates separately for each department

orderBy("salary") → defines the order inside each partition

In [27]:
#row_number()
#Assigns a sequential number to rows within the partition.

df.withColumn("row_number", row_number().over(windowSpec)).show()

+---+-------+----------+------+----------+
| id|   name|department|salary|row_number|
+---+-------+----------+------+----------+
|  8|   renu|   Finance|  4700|         1|
|  6|  Frank|   Finance|  4800|         2|
|  7|   renu|   Finance|  4900|         3|
|  4|  David|   Finance|  5200|         4|
|  1|  Alice|        HR|  5000|         1|
|  5|    Eva|        HR|  6000|         2|
|  2|    Bob|        IT|  6000|         1|
|  3|Charlie|        IT|  7000|         2|
+---+-------+----------+------+----------+



In [103]:
#rank()

#Ranks rows with gaps for ties.

df.withColumn("rank", rank().over(windowSpec)).show()

+---+-------+----------+------+----+
| id|   name|department|salary|rank|
+---+-------+----------+------+----+
|  4|  David|   Finance|  5200|   1|
|  6|  Frank|   Finance|  4800|   2|
|  7|   renu|   Finance|  4800|   2|
|  8|   renu|   Finance|  4700|   4|
|  5|    Eva|        HR|  6000|   1|
|  1|  Alice|        HR|  5000|   2|
|  3|Charlie|        IT|  7000|   1|
|  2|    Bob|        IT|  6000|   2|
+---+-------+----------+------+----+



In [104]:
#dense_rank()
#Ranks rows without gaps for ties.

df.withColumn("dense_rank", dense_rank().over(windowSpec)).show()

+---+-------+----------+------+----------+
| id|   name|department|salary|dense_rank|
+---+-------+----------+------+----------+
|  4|  David|   Finance|  5200|         1|
|  6|  Frank|   Finance|  4800|         2|
|  7|   renu|   Finance|  4800|         2|
|  8|   renu|   Finance|  4700|         3|
|  5|    Eva|        HR|  6000|         1|
|  1|  Alice|        HR|  5000|         2|
|  3|Charlie|        IT|  7000|         1|
|  2|    Bob|        IT|  6000|         2|
+---+-------+----------+------+----------+



# How to Run SQL Queries in PySpark?


In [40]:
data = [("Alice", 50), ("Bob", 30), ("Charlie", 40)]
columns = ["name", "score"]

df = spark.createDataFrame(data, columns)
df.show()

+-------+-----+
|   name|score|
+-------+-----+
|  Alice|   50|
|    Bob|   30|
|Charlie|   40|
+-------+-----+



In [41]:
#Register DataFrame as a Temporary View

#To run SQL queries, you need to register the DataFrame as a temporary view:

df.createOrReplaceTempView("students")

In [ ]:
#Why Use TempView?
Enables connection  between PySpark DataFrame ,API and SQL.
Easier for students who already know SQL.
Can join multiple DataFrames using SQL queries.
Helps in exploratory analysis quickly.


In [ ]:
#Run SQL Queries
#Now you can run SQL queries using spark.sql:

In [42]:
# Select all rows
spark.sql("SELECT * FROM students").show()

+-------+-----+
|   name|score|
+-------+-----+
|  Alice|   50|
|    Bob|   30|
|Charlie|   40|
+-------+-----+



In [43]:
# Filter with SQL
spark.sql("SELECT name, score FROM students WHERE score > 30").show()

+-------+-----+
|   name|score|
+-------+-----+
|  Alice|   50|
|Charlie|   40|
+-------+-----+



In [45]:
# Order by score
spark.sql("SELECT name, score FROM students ORDER BY score ").show()

+-------+-----+
|   name|score|
+-------+-----+
|    Bob|   30|
|Charlie|   40|
|  Alice|   50|
+-------+-----+



In [46]:
from pyspark.sql.functions import avg

# Using SQL
avg_score = spark.sql("SELECT AVG(score) as avg_score FROM students")
avg_score.show()

+---------+
|avg_score|
+---------+
|     40.0|
+---------+



# FINISH 